# 03 — Falsifiability harness

Load **your** multivariate CSV and run protocol defaults (window = 13).

**Labels:**
- Demo random walk below → `[OPERACIONAL]`
- Your real data → report outcomes as `[EMPÍRICO]` and follow `docs/EXPERIMENTAL_PROTOCOL.md`

Predictions list: `docs/FALSIFIABLE_PREDICTIONS.md` (P1–P4).


## Setup


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if (ROOT / "python" / "core").is_dir():
    pass
elif (ROOT.parent / "python" / "core").is_dir():
    ROOT = ROOT.parent
else:
    # walk up
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "python" / "core").is_dir():
            ROOT = p
            break

sys.path.insert(0, str(ROOT / "python"))
from core import (
    THETA_CHAOS, THETA_STABLE, DELTA,
    compute_taus, accumulate_time, gate_function,
)
print("ROOT =", ROOT)
print("δ =", DELTA, "τ_ch =", THETA_CHAOS, "τ_st =", THETA_STABLE)

## Loader and report


In [ ]:
def load_csv(path):
    path = Path(path)
    data = np.genfromtxt(path, delimiter=",")
    X = np.asarray(data, dtype=float)
    if X.ndim == 1:
        raise ValueError("Need at least 2 columns")
    mask = np.isfinite(X).all(axis=0)
    X = X[:, mask]
    if X.shape[1] < 2:
        raise ValueError("Need ≥2 finite numeric columns")
    return X

def report(X, label="data"):
    tg, _ = compute_taus(X, window_size=13)
    T, dtk, g, depth = accumulate_time(tg, window_size=13)
    valid = tg[~np.isnan(tg)]
    out = {
        "label": label,
        "shape": X.shape,
        "mean_tau": float(valid.mean()),
        "std_tau": float(valid.std()),
        "frac_stable": float(np.mean(valid >= THETA_STABLE)),
        "frac_chaos": float(np.mean(np.abs(valid) < THETA_CHAOS)),
        "frac_anti": float(np.mean(valid <= -THETA_CHAOS)),
        "T_end": float(T[-1]),
        "max_k": int(depth.max()),
        "tau": tg,
        "T": T,
        "g": g,
    }
    print(f"Dataset: {label}  shape={X.shape}")
    print(f"  mean τₛ={out['mean_tau']:.4f}  std={out['std_tau']:.4f}")
    print(f"  regimes: stable={out['frac_stable']:.2%} chaos={out['frac_chaos']:.2%} anti={out['frac_anti']:.2%}")
    print(f"  T_RECD end={out['T_end']:.6f}  max depth k={out['max_k']}")
    print("  See docs/FALSIFIABLE_PREDICTIONS.md and docs/EXPERIMENTAL_PROTOCOL.md §7")
    return out

## Demo: multivariate random walk `[OPERACIONAL]`


In [ ]:
rng = np.random.default_rng(0)
X_demo = np.cumsum(rng.normal(size=(300, 4)), axis=0)
out = report(X_demo, "demo_random_walk")

fig, ax = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
t = np.arange(len(out["tau"]))
ax[0].plot(t, out["tau"], lw=0.8)
ax[0].axhline(THETA_CHAOS, color="C3", ls="--", lw=0.8)
ax[0].axhline(-THETA_CHAOS, color="C3", ls="--", lw=0.8)
ax[0].set_ylabel("τₛ")
ax[0].set_title("Falsifiability demo — random walk")
ax[1].plot(t, out["g"], lw=0.8, color="C1")
ax[1].set_ylabel("g")
ax[2].plot(t, out["T"], lw=1.0, color="C4")
ax[2].set_ylabel("T_RECD")
ax[2].set_xlabel("time index")
fig.tight_layout()
plt.show()

## Your data

Set `CSV_PATH` to a file with shape `(T, N)`, `N ≥ 2`, and run the next cell.

```python
CSV_PATH = ROOT / "path" / "to" / "your.csv"  # edit me
out = report(load_csv(CSV_PATH), str(CSV_PATH))
```

Open a GitHub issue with label `contradiction` if a prediction P1–P4 fails under the protocol.


In [ ]:
# Optional: uncomment and set path
# CSV_PATH = ROOT / "data" / "your.csv"
# out_user = report(load_csv(CSV_PATH), str(CSV_PATH))
print("Set CSV_PATH above to analyze your series.")